In [ ]:
import os
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')



In [ ]:
PROJECT_PATH = '/content/drive/MyDrive/ecommerce-growth-marketing-analytics'
RAW_PATH = f'{PROJECT_PATH}/data/raw'
CLEAN_PATH = f'{PROJECT_PATH}/data/cleaned'
os.listdir(RAW_PATH)

In [ ]:
orders = pd.read_csv(f'{RAW_PATH}/olist_orders_dataset.csv')

In [ ]:
orders.info()

In [ ]:
orders.head()

In [ ]:
orders.isnull().sum()

In [ ]:
orders['order_status'].value_counts()

In [ ]:
orders['order_id'].duplicated().sum()

In [ ]:
date_columns=['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']
for column in date_columns:
  orders[column]=pd.to_datetime(orders[column])
orders.info()

In [ ]:
orders[(orders['order_status']=='delivered') & (orders['order_delivered_customer_date'].isnull())]

In [ ]:
orders[orders['order_delivered_customer_date']< orders['order_purchase_timestamp']]


In [ ]:
orders[
    orders['order_delivered_customer_date']
    < orders['order_delivered_carrier_date']
]

In [ ]:
orders[orders['order_approved_at'] < orders['order_purchase_timestamp']]

In [ ]:
orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date']
    - orders['order_estimated_delivery_date']
).dt.days

orders['delivery_delay_days'].describe()

In [ ]:
orders['is_late'] = orders['delivery_delay_days'] > 0

orders.loc[orders['order_delivered_customer_date'].notna(),'is_late'].value_counts()

In [ ]:
late_delivery_rate = (orders.loc[orders['order_delivered_customer_date'].notna(),'is_late'].mean() * 100)

print(f"Late Delivery Rate: {late_delivery_rate:.2f}%")

In [ ]:
orders['delivery_time_days'] = (
    orders['order_delivered_customer_date']
    - orders['order_purchase_timestamp']
).dt.days

orders['delivery_time_days'].describe()

In [ ]:
print("First order:", orders['order_purchase_timestamp'].min())
print("Last order:", orders['order_purchase_timestamp'].max())

In [ ]:
orders['purchase_year'] = orders['order_purchase_timestamp'].dt.year
orders['purchase_month'] = orders['order_purchase_timestamp'].dt.month
orders['purchase_year_month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)

orders[
    ['order_purchase_timestamp', 'purchase_year',
     'purchase_month', 'purchase_year_month']
].head()

In [ ]:
orders.shape

In [ ]:
orders.to_csv(f'{CLEAN_PATH}/olist_orders_cleaned.csv',index=False)

In [ ]:
os.path.exists(f'{CLEAN_PATH}/olist_orders_cleaned.csv')

In [ ]:
## 2. Customers Dataset Cleaning

In [ ]:
customers = pd.read_csv(f'{RAW_PATH}/olist_customers_dataset.csv')

customers.head()

In [ ]:
customers.info()

In [ ]:
print("Duplicate customer_id:",
      customers['customer_id'].duplicated().sum())

print("Duplicate customer_unique_id:",
      customers['customer_unique_id'].duplicated().sum())

In [ ]:
customers['customer_unique_id'].nunique()

In [ ]:
customers['customer_state'].value_counts()
#customers['customer_city'].value_counts()
print("Unique cities:", customers['customer_city'].nunique())

customers['customer_city'].value_counts().head(10)

In [ ]:
orders_customer_check = orders['customer_id'].isin(
    customers['customer_id'])

print("Matched customer IDs:", orders_customer_check.sum())
print("Unmatched customer IDs:", (~orders_customer_check).sum())

In [ ]:
customers.to_csv(
    f'{CLEAN_PATH}/olist_customers_cleaned.csv',
    index=False)

os.path.exists(f'{CLEAN_PATH}/olist_customers_cleaned.csv')

In [ ]:
## 3. Order Items Dataset Cleaning

In [ ]:
order_items = pd.read_csv(f'{RAW_PATH}/olist_order_items_dataset.csv')

order_items.head()

In [ ]:
order_items.info()

In [ ]:
order_items.duplicated(subset=['order_id', 'order_item_id']).sum()

In [ ]:
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

order_items['shipping_limit_date'].dtype

In [ ]:
order_items[['price', 'freight_value']].describe()

In [ ]:
print("Negative prices:", (order_items['price'] < 0).sum())
print("Negative freight values:", (order_items['freight_value'] < 0).sum())

In [ ]:
order_items_order_check = order_items['order_id'].isin(orders['order_id'])

print("Matched order items:", order_items_order_check.sum())
print("Unmatched order items:", (~order_items_order_check).sum())

In [ ]:
## Products Dataset Cleaning

In [ ]:
products= pd.read_csv(f'{RAW_PATH}/olist_products_dataset.csv')

product_check = order_items['product_id'].isin(
    products['product_id'])

print("Matched product IDs:", product_check.sum())
print("Unmatched product IDs:", (~product_check).sum())

In [ ]:
##  sellers Dataset Cleaning

In [ ]:
sellers = pd.read_csv(f'{RAW_PATH}/olist_sellers_dataset.csv')

seller_check = order_items['seller_id'].isin(
    sellers['seller_id'])

print("Matched seller IDs:", seller_check.sum())
print("Unmatched seller IDs:", (~seller_check).sum())

In [ ]:
items_per_order = order_items.groupby('order_id')['order_item_id'].max()

items_per_order.describe()

In [ ]:
order_items['item_total_value'] = (order_items['price'] + order_items['freight_value'])

order_items[['price', 'freight_value', 'item_total_value']].head()

In [ ]:
order_items.to_csv(f'{CLEAN_PATH}/olist_order_items_cleaned.csv',index=False)
os.path.exists(f'{CLEAN_PATH}/olist_order_items_cleaned.csv')

In [ ]:
## 4. Payments Dataset Cleaning

In [ ]:
payments = pd.read_csv(f'{RAW_PATH}/olist_order_payments_dataset.csv')
payments.head()

In [ ]:
payments.info()

In [ ]:
payments.duplicated(
    subset=['order_id', 'payment_sequential']
).sum()

In [ ]:
payments['payment_type'].value_counts()


In [ ]:
payments[
    payments['payment_type'] == 'not_defined'
]

In [ ]:
payments['payment_value'].describe()

In [ ]:
print(
    "Negative payment values:",
    (payments['payment_value'] < 0).sum()
)

print(
    "Zero payment values:",
    (payments['payment_value'] == 0).sum()
)

In [ ]:
payments[payments['payment_value'] == 0]

In [ ]:
payment_order_check = payments['order_id'].isin(['order_id'])

print("Matched payments:", payment_order_check.sum())
print("Unmatched payments:", (~payment_order_check).sum())

In [ ]:
print("Orders shape:", orders.shape)
print("Orders columns:", orders.columns.tolist())

print("\nFirst 3 order IDs:")
print(orders['order_id'].head(3).tolist())

print("\nFirst 3 payment order IDs:")
print(payments['order_id'].head(3).tolist())

In [ ]:
test_id = payments['order_id'].iloc[0]

print("Payment order ID:", test_id)
print("Exists in Orders:", test_id in orders['order_id'].values)

In [ ]:
payment_order_check = payments['order_id'].isin(orders['order_id'])

print("Matched payments:", payment_order_check.sum())
print("Unmatched payments:", (~payment_order_check).sum())

In [ ]:
payments['payment_installments'].describe()

print("Negative installments:",(payments['payment_installments'] < 0).sum())

print("Zero installments:",(payments['payment_installments'] == 0).sum())

In [ ]:
payments[payments['payment_installments'] == 0]

In [ ]:
payments.to_csv(
    f'{CLEAN_PATH}/olist_order_payments_cleaned.csv',
    index=False
)

os.path.exists(
    f'{CLEAN_PATH}/olist_order_payments_cleaned.csv'
)

In [ ]:
## 5. Reviews Dataset Cleaning

In [ ]:
reviews = pd.read_csv(f'{RAW_PATH}/olist_order_reviews_dataset.csv')

reviews.head()

In [ ]:
reviews.info()

In [ ]:
reviews['review_score'].value_counts().sort_index()

In [ ]:
print("Duplicate review_id:",reviews['review_id'].duplicated().sum())

print("Duplicate order_id:",reviews['order_id'].duplicated().sum())

In [ ]:
duplicate_review_ids = reviews[
    reviews['review_id'].duplicated(keep=False)
].sort_values(['review_id', 'review_answer_timestamp'])

duplicate_review_ids.head(20)

In [ ]:
reviews.duplicated().sum()

In [ ]:
duplicate_order_reviews = reviews[reviews['order_id'].duplicated(keep=False)].sort_values(['order_id', 'review_answer_timestamp'])

duplicate_order_reviews.head(20)

In [ ]:
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])

reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

reviews[['review_creation_date', 'review_answer_timestamp']].dtypes

In [ ]:
review_order_check = reviews['order_id'].isin(orders['order_id'])

print("Matched reviews:", review_order_check.sum())
print("Unmatched reviews:", (~review_order_check).sum())

In [ ]:
invalid_review_timing = (reviews['review_answer_timestamp']< reviews['review_creation_date']).sum()

print("Reviews answered before creation:", invalid_review_timing)

In [ ]:
print("Missing review titles:", reviews['review_comment_title'].isna().sum())
print("Missing review messages:", reviews['review_comment_message'].isna().sum())

In [ ]:
print("Reviews with no written comment:",(reviews['review_comment_title'].isna()& reviews['review_comment_message'].isna()).sum())

In [ ]:
reviews.to_csv(f'{CLEAN_PATH}/olist_order_reviews_cleaned.csv',index=False)

os.path.exists(f'{CLEAN_PATH}/olist_order_reviews_cleaned.csv')

In [ ]:
## 6. Products Dataset Cleaning

In [ ]:
products = pd.read_csv(f'{RAW_PATH}/olist_products_dataset.csv')

print(products.head())
products.info()

In [ ]:
print("Duplicate product_id:",products['product_id'].duplicated().sum())

print("Unique products:",products['product_id'].nunique())

In [ ]:
missing_category_products = products[products['product_category_name'].isna()]
missing_category_products[
    [
        'product_category_name',
        'product_name_lenght',
        'product_description_lenght',
        'product_photos_qty'
    ]
].isna().sum()

In [ ]:
products[
    products['product_weight_g'].isna()
    | products['product_length_cm'].isna()
    | products['product_height_cm'].isna()
    | products['product_width_cm'].isna()]

In [ ]:
numeric_columns = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

for column in numeric_columns:
    print(
        column,
        "zero or negative:",
        (products[column] <= 0).sum()
    )

In [ ]:
products[
    products['product_weight_g'] <= 0][
    [
        'product_id',
        'product_category_name',
        'product_weight_g',
        'product_length_cm',
        'product_height_cm',
        'product_width_cm'
    ]
]

In [ ]:
products.loc[
    products['product_weight_g'] == 0,
    'product_weight_g'
] = pd.NA

products['product_weight_g'].isna().sum()

In [ ]:
category_translation = pd.read_csv(
    f'{RAW_PATH}/product_category_name_translation.csv'
)

category_check = products[
    'product_category_name'
].dropna().isin(
    category_translation['product_category_name']
)

print("Matched product categories:", category_check.sum())
print("Unmatched product categories:", (~category_check).sum())

In [ ]:
unmatched_categories = products.loc[
    products['product_category_name'].notna()
    & ~products['product_category_name'].isin(
        category_translation['product_category_name']
    ),
    'product_category_name'
].value_counts()

unmatched_categories

In [ ]:
category_translation[
    category_translation['product_category_name'].str.contains(
        'pc|cozinha|alimentos',
        case=False,
        na=False
    )
]

In [ ]:
products.to_csv(
    f'{CLEAN_PATH}/olist_products_cleaned.csv',
    index=False
)

os.path.exists(
    f'{CLEAN_PATH}/olist_products_cleaned.csv'
)

In [ ]:
## 7. Sellers Dataset Cleaning

In [ ]:
sellers = pd.read_csv(f'{RAW_PATH}/olist_sellers_dataset.csv')

sellers.head()

In [ ]:
sellers.info()

In [ ]:
print(
    "Duplicate seller_id:",
    sellers['seller_id'].duplicated().sum()
)

print(
    "Unique sellers:",
    sellers['seller_id'].nunique()
)

In [ ]:
sellers['seller_state'].value_counts().sort_index()

In [ ]:
seller_check = order_items['seller_id'].isin(
    sellers['seller_id']
)

print("Matched seller records:", seller_check.sum())
print("Unmatched seller records:", (~seller_check).sum())

In [ ]:
sellers.to_csv(
    f'{CLEAN_PATH}/olist_sellers_cleaned.csv',
    index=False
)

os.path.exists(
    f'{CLEAN_PATH}/olist_sellers_cleaned.csv'
)

In [ ]:
## 8. Category Translation Dataset Cleaning

In [98]:
category_translation = pd.read_csv(f'{RAW_PATH}/product_category_name_translation.csv')

print(category_translation.head())
category_translation.info()



    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto
3         cama_mesa_banho                bed_bath_table
4        moveis_decoracao               furniture_decor
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


In [99]:
print("Duplicate Portuguese categories:",category_translation['product_category_name'].duplicated().sum())

print("Duplicate English categories:",category_translation['product_category_name_english'].duplicated().sum())

Duplicate Portuguese categories: 0
Duplicate English categories: 0


In [100]:
category_translation.to_csv(
    f'{CLEAN_PATH}/product_category_name_translation_cleaned.csv',
    index=False
)

os.path.exists(
    f'{CLEAN_PATH}/product_category_name_translation_cleaned.csv'
)

True

In [ ]:
## 9. Geolocation Dataset Cleaning

In [103]:
geolocation = pd.read_csv(f'{RAW_PATH}/olist_geolocation_dataset.csv')

print(geolocation.head())
geolocation.info()

   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1037       -23.545621       -46.639292   
1                         1046       -23.546081       -46.644820   
2                         1046       -23.546129       -46.642951   
3                         1041       -23.544392       -46.639499   
4                         1035       -23.541578       -46.641607   

  geolocation_city geolocation_state  
0        sao paulo                SP  
1        sao paulo                SP  
2        sao paulo                SP  
3        sao paulo                SP  
4        sao paulo                SP  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   

In [104]:
print("Exact duplicate rows:",geolocation.duplicated().sum())

Exact duplicate rows: 261831


In [105]:
geolocation = geolocation.drop_duplicates().copy()

print("Rows after removing exact duplicates:", len(geolocation))
print("Exact duplicate rows remaining:", geolocation.duplicated().sum())

Rows after removing exact duplicates: 738332
Exact duplicate rows remaining: 0


In [106]:
print(
    "Invalid latitude rows:",
    ((geolocation['geolocation_lat'] < -90) |
     (geolocation['geolocation_lat'] > 90)).sum()
)

print(
    "Invalid longitude rows:",
    ((geolocation['geolocation_lng'] < -180) |
     (geolocation['geolocation_lng'] > 180)).sum()
)

Invalid latitude rows: 0
Invalid longitude rows: 0


In [107]:
customer_geo_check = customers[
    'customer_zip_code_prefix'
].isin(
    geolocation['geolocation_zip_code_prefix']
)

print("Matched customer ZIP records:", customer_geo_check.sum())
print("Unmatched customer ZIP records:", (~customer_geo_check).sum())

Matched customer ZIP records: 99163
Unmatched customer ZIP records: 278


In [108]:
seller_geo_check = sellers[
    'seller_zip_code_prefix'
].isin(
    geolocation['geolocation_zip_code_prefix']
)

print("Matched seller ZIP records:", seller_geo_check.sum())
print("Unmatched seller ZIP records:", (~seller_geo_check).sum())

Matched seller ZIP records: 3088
Unmatched seller ZIP records: 7


In [109]:
geolocation.to_csv(
    f'{CLEAN_PATH}/olist_geolocation_cleaned.csv',
    index=False
)

os.path.exists(
    f'{CLEAN_PATH}/olist_geolocation_cleaned.csv'
)

True